# Session 4

![](images/hail_mary_lab_illustration.png)

## Mini challenge: the Hail Mary gene hunt

You are Dr. Grace Ryland, and humanity's last, most fragile plan is riding on you. You are the scientist aboard the *Hail Mary* — the ship [Project Hail Mary](https://www.google.com/search?q=project+hail+mary&oq=projec&gs_lcrp=EgZjaHJvbWUqDQgAEAAY4wIYsQMYgAQyDQgAEAAY4wIYsQMYgAQyCggBEC4YsQMYgAQyBggCEEUYOTINCAMQLhiDARixAxiABDINCAQQLhjHARjRAxiABDIGCAUQRRg8MgYIBhBFGEEyBggHEEUYPNIBCDQ2MDlqMGo3qAIAsAIA&sourceid=chrome&source=chrome.ob&ie=UTF-8) sent out to figure out why the sun is dying, and why it might be about to take every other star in a 50-lightyear bubble down with it.

Somewhere along the way you picked up an unlikely lab partner: [Rocky](https://www.orzgk.com/product/panda-studio-rocky-project-hail-mary/), an alien engineer with five arms, a knack for improvising equipment out of scrap, and absolutely no sense of how fragile human sample-labelling systems are. During a rough bit of turbulence, Rocky knocked over a rack of 10 patient samples. The labels are gone. The DNA, RNA and protein are all still there — you just don't know whose is whose anymore.

This is, to put it mildly, a problem. Back on Earth, these 10 patients are relying on this exact data. And thankfully, you're not starting from nothing: you have each sample's full multi-omics profile, and you have the clinical notes you scribbled down before the accident — the "amaze, amaze, amaze, scientific notes" that are about to earn their keep.

Use the patient histories below, the molecular signatures locked in each sample, and every trick you've picked up so far in this workshop to re-identify all 10 samples. One piece of good news from earlier statistics work: you already know the mislabelled batch is exactly 5 Luminal A and 5 Luminal B subtypes — so whatever you land on, it should split 5/5.

*Godspeed, Dr. Ryland. Earth is listening.*

### How this notebook is organised

1. **Setup** — environment check + imports
2. **Load data** — the 10 mislabelled ("challenge") samples, the labelled training cohort, and a pretrained MOFA multi-omics model
3. **Project** the challenge samples into the MOFA factor space learned from the training cohort
4. **Visualise** the factor space to see where the unknown samples fall relative to known subtypes
5. **Classify** each unknown sample as Luminal A / Luminal B using a simple logistic regression on the factors
6. **Explain** each prediction by finding the genes driving each unknown sample's factor values, then linking those genes to diseases via a knowledge graph — to match against the clinical notes below
7. **Answer** — fill in your final `Pat_i -> TCGA-xxxx` mapping

## Patient clinical notes

Prior to the accident, these are the disease/phenotype notes you recorded for each of the 10 patients whose samples are now mislabelled `Pat_0` ... `Pat_9`. You'll use these — together with the molecular evidence gathered below — to work out which `Pat_i` is which `TCGA-xxxx`.

**TCGA-EW-A6S9** — Presented with invasive breast carcinoma. Immunohistochemistry came back HER2-negative, but strongly positive for both estrogen and progesterone receptors, marking this out as a clearly hormone-receptor-driven tumor.

**TCGA-A2-A0EP** — Biopsy of a breast mass confirmed adenocarcinoma of ductal origin — glandular architecture arising from the mammary ducts, the most textbook presentation of the disease.

**TCGA-E2-A1IG** — A patient managing both asthma and Parkinson disease for several years, whose case took a sharp turn with the diagnosis of breast angiosarcoma, a rare and fast-growing vascular malignancy rather than a typical epithelial breast cancer.

**TCGA-AQ-A1H3** — Decades of type 1 diabetes had already taken a toll by the time she came in: coronary artery disease and chronic kidney disease, the two classic downstream complications of long-standing diabetic vascular damage.

**TCGA-AC-A2FE** — Type 2 diabetes was already on the chart when breast angiosarcoma was diagnosed — a second, unrelated but noteworthy vascular malignancy layered onto an existing metabolic condition.

**TCGA-A2-A1FX** — Screening first picked up lobular neoplasia; further workup of a separate area of the breast then revealed medullary carcinoma, a less common but relatively well-differentiated histological subtype.

**TCGA-A2-A4S3** — Living with multiple sclerosis, this patient went on to develop two independent primary cancers around the same time: an adenocarcinoma of the lung and a colorectal cancer, an unusually heavy cancer burden for one case.

**TCGA-C8-A26W** — Chronic kidney disease was a long-standing issue by the time gastric carcinoma appeared, and the picture was complicated further still by a later diagnosis of acute myeloid leukemia — three serious conditions converging in one patient.

**TCGA-AR-A2LK** — A palpable lump led to biopsy and a diagnosis of ductal adenocarcinoma of the breast, the same core histology as the most common form of breast adenocarcinoma.

**TCGA-AO-A0JD** — Type 2 diabetes was a pre-existing condition when this patient was diagnosed with breast angiosarcoma, a rare vascular tumor of the breast rather than the far more common ductal or lobular types.

## 1. Setup

In [ ]:
# Quick environment check (optional).
# Confirms the notebook is running on the expected machine / Python env / folder,
# since path or environment mismatches are a common source of confusing errors.
import socket
import os
import sys

print("Host:", socket.gethostname())
print("Python:", sys.executable)
print("Working dir:", os.getcwd())

In [ ]:
# ── Standard library ─────────────────────────────────────────────────────────
from pathlib import Path
import pickle

# ── Third-party ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from matplotlib.colors import TwoSlopeNorm
from IPython.display import display

from sklearn.linear_model import LogisticRegression
# Extra metrics you may want once you've settled on a final answer and want to
# sanity-check a prediction against a partially-known label, or during model
# development on the labelled training cohort:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report

# MOFA (Multi-Omics Factor Analysis): mofax is used to *read* a pretrained model.
# mofapy2 is only needed if you want to *train* a MOFA model from scratch, which
# isn't required for this challenge (a pretrained model is provided below).
import mofax as mfx

# ── Custom helpers for this workshop (see s4_helpers.py) ───────────────────────
from s4_helpers import (
    load_omics,
    evaluate_predictions,
    load_kg,
    print_graph_info,
    map_genes_to_kg,
    diseases_for_genes,
)

# ── Reproducibility ───────────────────────────────────────────────────────────
RANDOM_STATE = 42

## 2. Load data

We need three things:

1. The **10 mislabelled "challenge" samples** (`Pat_0` ... `Pat_9`) — these are the ones we're re-identifying.
2. The **labelled training cohort** used to train the MOFA model, plus the train/test patient ID split that was used at training time.
3. The **pretrained MOFA model** itself, so we can project the challenge samples into the same factor space.

In [ ]:
# The 10 mislabelled samples for this challenge.
DATA_DIR = Path(".")

X_challenge_omics, y_challenge = load_omics(
    DATA_DIR,
    omic_filename="challenge_omics",
    omic_keys=["transcriptomics", "proteomics", "methylation"],
)

# The TCGA IDs Rocky mixed up (order not meaningful — this is what we're solving for).
missing_ids = [
    "TCGA-EW-A6S9", "TCGA-A2-A0EP", "TCGA-E2-A1IG",
    "TCGA-AQ-A1H3", "TCGA-AC-A2FE", "TCGA-A2-A1FX",
    "TCGA-A2-A4S3", "TCGA-C8-A26W", "TCGA-AR-A2LK", "TCGA-AO-A0JD",
]
pat_ids = [f"Pat_{i}" for i in range(10)]

# The true subtype of each challenge sample is unknown by construction — that's
# half of what we're inferring (the other half is the exact patient identity).
y_challenge["subtype"] = ["Unknown" for _ in range(10)]
y_challenge = y_challenge[["subtype"]]

# Relabel the challenge cohort with anonymised Pat_0..Pat_9 IDs.
for view in X_challenge_omics:
    X_challenge_omics[view].index = pat_ids
y_challenge.index = pat_ids

X_challenge_omics

In [ ]:
# The labelled training cohort (TCGA-BRCA) that the MOFA model below was trained on.
TRAIN_DATA_DIR = Path("/data/")

X_omics, y = load_omics(
    TRAIN_DATA_DIR,
    omic_filename="omics",
    omic_keys=["transcriptomics", "proteomics", "methylation"],
)

with open(TRAIN_DATA_DIR / "patient_splits.pkl", "rb") as f:
    splits = pickle.load(f)

train_ids = splits["train_ids"]
test_ids = splits["test_ids"]

# Split the labels using the same patient IDs used for the omics data.
y_train = y.loc[train_ids]
y_test = y.loc[test_ids]

In [ ]:
# Load the pretrained MOFA model.
mofa_model_mfx = mfx.mofa_model("mofa_pretrained.hdf5")

print(mofa_model_mfx)          # Model overview
print(mofa_model_mfx.shape)    # (n_samples, n_factors)

# Extract key components.
factors = mofa_model_mfx.get_factors(df=True)               # Factor/latent space values
weights = mofa_model_mfx.get_weights(df=True)         # Feature weights/loadings
var_exp = mofa_model_mfx.get_variance_explained()     # R^2 per factor per view

# NOTE: the exact factor-name format (e.g. "Factor1" vs "Factor 1") depends on
# your mofax version. Check this before relying on either format later in the
# notebook:
print("Factor column names look like:", list(factors.columns[:3]))

## 3. Project the challenge samples into the MOFA factor space

MOFA was fit on the training cohort only. To place the 10 unknown challenge
samples on the same latent axes, we need to *project* them using the feature
weights the model already learned — without ever touching the (unknown) labels.

In [ ]:
# Restrict each omics view to the features MOFA actually used (its most-variable,
# training-selected feature set), so train and test matrices line up column-for-column.
X_train_raw = {name: X.loc[train_ids] for name, X in X_omics.items()}
X_test_raw = {name: X for name, X in X_challenge_omics.items()}

features_in_mofa = set(weights.index)
X_train_omics = {}
X_test_omics = {}

"""
------------------
YOUR CODE HERE
------------------
1. Loop over each omics view in X_train_raw

2. Find which features are shared between MOFA and the current view

3. Subset the train data to only those features

4. Subset the test data to the same features
"""

print("Feature counts after filtering to MOFA's feature set:")
for name in X_omics:
    print(
        f"  {name:15s}: {X_omics[name].shape[1]:6d} original -> "
        f"{X_train_omics[name].shape[1]:6d} used by MOFA"
    )

In [ ]:
def project_test_patients_to_mofa_factors(model, X_train_by_view, X_test_by_view, train_factors, view_names):
    """Project held-out patients into the fixed MOFA factor space.

    The projection uses MOFA weights learned from training patients. To keep
    train and test factor values comparable, preprocessing and calibration are
    fit on training patients only and then applied unchanged to test patients
    (test labels are never used anywhere in this function).
    """
    factor_columns = train_factors.columns.astype(str).tolist()
    projected_test_by_view = []

    for view_name in view_names:
        # W is the learned feature x factor weight matrix for this omics view.
        view_weights = model.get_weights(views=view_name, df=True)
        view_weights.columns = view_weights.columns.astype(str)
        view_weights = view_weights.reindex(columns=factor_columns)

        # Use only features present in the trained weights AND both train/test matrices.
        common_features = view_weights.index.intersection(X_train_by_view[view_name].columns)
        common_features = common_features.intersection(X_test_by_view[view_name].columns)
        view_weights = view_weights.loc[common_features]

        X_train_view = X_train_by_view[view_name].loc[:, common_features].astype(float)
        X_test_view = X_test_by_view[view_name].loc[:, common_features].astype(float)
        X_train_view.index = X_train_view.index.astype(str)
        X_test_view.index = X_test_view.index.astype(str)

        # Learn centering/scaling on training patients only, then apply it to test patients.
        train_mean = X_train_view.mean(axis=0)
        train_std = X_train_view.std(axis=0, ddof=0).replace(0, 1)
        X_train_scaled = (X_train_view - train_mean) / train_std
        X_test_scaled = (X_test_view - train_mean) / train_std

        # Project X into the fixed MOFA weight space via the pseudo-inverse of W.
        raw_train_projection = X_train_scaled.to_numpy() @ np.linalg.pinv(view_weights.to_numpy()).T
        raw_test_projection = X_test_scaled.to_numpy() @ np.linalg.pinv(view_weights.to_numpy()).T

        # Calibrate the raw projection to the factor scale returned by the trained
        # MOFA model. Fitted on training patients only; test labels are never used.
        train_design = np.column_stack([raw_train_projection, np.ones(raw_train_projection.shape[0])])
        test_design = np.column_stack([raw_test_projection, np.ones(raw_test_projection.shape[0])])
        train_target = train_factors.loc[X_train_view.index, factor_columns].to_numpy()
        calibration = np.linalg.lstsq(train_design, train_target, rcond=None)[0]
        projected_values = test_design @ calibration

        projected = pd.DataFrame(projected_values, index=X_test_view.index, columns=factor_columns)
        projected_test_by_view.append(projected)

    # Each view gives one projected factor table; average them since all views
    # are available for every challenge sample here.
    return sum(projected_test_by_view) / len(projected_test_by_view)

In [ ]:
# Extract training-patient factor values straight from the fitted MOFA model.
train_factors_mfx = mofa_model_mfx.get_factors(df=True)
train_factors_mfx.index = train_factors_mfx.index.astype(str)

# Project the 10 held-out challenge patients into that same factor space.
test_factors_mfx = project_test_patients_to_mofa_factors(
    mofa_model_mfx,
    X_train_omics,
    X_test_omics,
    train_factors_mfx,
    list(X_train_omics.keys()),
)

In [ ]:
# Attach the known training subtypes to the model so mofax's built-in plots can
# colour training points by subtype.
mofa_sample_metadata = pd.DataFrame(index=train_factors_mfx.index)
mofa_sample_metadata["subtype"] = y_train.reindex(train_factors_mfx.index).values

mofa_model_mfx.samples_metadata = mofa_sample_metadata

## 4. Visualise the factor space

In [ ]:
# Built-in mofax scatter grid, training patients only, coloured by subtype.
mfx.plot_factors_scatter(
    mofa_model_mfx,
    x="Factor1",
    y="Factor2",
    group_label="subtype",
    color="subtype",
    zero_line_x=True,
    zero_line_y=True,
    ncols=3,
    size=40,
    alpha=1,
)
plt.show()

In [ ]:
# Combine training (known subtype) and challenge (unknown subtype) factors into
# one long table we can reuse for every plot below.
plot_df = pd.concat([train_factors_mfx, test_factors_mfx])

subtypes_col = pd.concat([
    y_train,
    pd.Series(["Unknown"] * len(test_factors_mfx), index=test_factors_mfx.index),
])
plot_df["subtype"] = subtypes_col

plot_df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

# Split known vs unknown subtypes (handles both NaN and the literal 'Unknown' label).
known = plot_df[plot_df["subtype"].notna() & ~plot_df["subtype"].isin(["Unknown"])]
unknown = plot_df[plot_df["subtype"].isna() | plot_df["subtype"].isin(["Unknown"])]

# Known subtypes as coloured circles.
sns.scatterplot(data=known, x="Factor1", y="Factor2", hue="subtype", s=40, alpha=1, ax=ax)

# Unknown (challenge) samples as bigger grey crosses, drawn on top.
ax.scatter(
    unknown["Factor1"], unknown["Factor2"],
    marker="X", s=120, color="grey", label="Unknown", zorder=5, linewidths=1.2,
)

ax.axhline(0, color="grey", linewidth=0.8, linestyle="--")
ax.axvline(0, color="grey", linewidth=0.8, linestyle="--")
ax.set_title("Factor 1 vs Factor 2 by Subtype")
ax.legend(title="Subtype", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

### 4.1 Compare subtypes across a wider set of factors

Two factors aren't always enough to separate every subtype — below we look at a
few more factors as boxplots (by known subtype) and then as individual scatter
points for the unknown samples, so we can eyeball which subtype each challenge
sample is likely to belong to before formally classifying it in Section 5.

In [ ]:
subtype_order = ["Basal", "LumB", "Her2", "LumA", "Normal"]
factors_of_interest = ["Factor2", "Factor1", "Factor6", "Factor4"]

fig, axes = plt.subplots(1, len(factors_of_interest), figsize=(15, 4), squeeze=False)

for ax, factor in zip(axes.ravel(), factors_of_interest):
    groups = [
        plot_df.loc[plot_df["subtype"] == subtype, factor].dropna()
        for subtype in subtype_order
    ]
    ax.boxplot(groups, tick_labels=subtype_order, showfliers=False)
    ax.set_title(factor)
    ax.set_ylabel("Factor value")
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Y-axis limits per factor, just for a bit of visual polish.
ylims = {
    "Factor2": (-8, 4),
    "Factor1": (-4, 4),
    "Factor6": (-4, 4),
    "Factor4": (-4, 6),
}

unknown_df = plot_df[plot_df["subtype"] == "Unknown"]

fig, axes = plt.subplots(1, len(factors_of_interest), figsize=(16, 5), squeeze=False)

for ax, factor in zip(axes.ravel(), factors_of_interest):
    vals = unknown_df[factor].dropna()

    ax.scatter(range(len(vals)), vals.values, s=80, color="steelblue", zorder=5)
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels(vals.index, rotation=90, fontsize=8)
    ax.axhline(0, color="grey", linewidth=0.8, linestyle="--")
    ax.set_ylim(ylims[factor])
    ax.set_title(factor)
    ax.set_ylabel("Factor value")
    ax.set_xlabel("Sample ID")

plt.tight_layout()
plt.show()

In [ ]:
# Built-in mofax heatmap of the mean factor value per subtype (training patients only —
# challenge patients have subtype == NaN/'Unknown' so they don't contribute here).
mfx.plot_factors_matrix(
    mofa_model_mfx,
    group_label="subtype",
    cmap="coolwarm",
    center=0,
)
plt.title("Mean MOFA factor value by subtype")
plt.show()

In [ ]:
# Recreate that same subtype-mean heatmap side-by-side with the unknown samples,
# on a shared colour scale, so it's easy to visually match each unknown sample's
# row-pattern to the subtype it most resembles.
heatmap_factors = [f"Factor{i}" for i in [1, 2, 4, 6]]

unknown_mask = plot_df["subtype"].isna() | plot_df["subtype"].isin(["Unknown", "NA", "unknown"])

unknown_mat = plot_df.loc[unknown_mask, heatmap_factors].dropna(how="all")
heat_unknown = unknown_mat.T  # rows = factors, cols = sample IDs

means_by_subtype = (
    plot_df.loc[~unknown_mask, ["subtype"] + heatmap_factors]
    .dropna(subset=["subtype"])
    .groupby("subtype")[heatmap_factors]
    .mean()
    .reindex(subtype_order)
)

# Colour scale is defined ONLY from the subtype-mean matrix, so the unknown
# heatmap is shown on the exact same scale as the reference panel.
vmax = np.nanmax(np.abs(means_by_subtype.values))
norm = TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)

fig = plt.figure(figsize=(max(10, 0.35 * heat_unknown.shape[1] + 6), 8))
gs = fig.add_gridspec(1, 3, width_ratios=[2.2, 0.18, max(4, 0.35 * heat_unknown.shape[1])], wspace=0.25)

ax_ref = fig.add_subplot(gs[0, 0])
cax = fig.add_subplot(gs[0, 1])
ax_unk = fig.add_subplot(gs[0, 2])

sns.heatmap(means_by_subtype.T, ax=ax_ref, cmap="coolwarm", norm=norm, cbar=False,
            linewidths=0.2, linecolor="white")
ax_ref.set_title("Mean MOFA factor value by subtype")
ax_ref.set_xlabel("Factors")
ax_ref.set_ylabel("Subtype")
ax_ref.tick_params(axis="x", rotation=90)

sns.heatmap(heat_unknown, ax=ax_unk, cmap="coolwarm", norm=norm, cbar=True,
            cbar_ax=cax, linewidths=0.0)
ax_unk.set_title("Unknown participants (same scale as subtype means)")
ax_unk.set_xlabel("Sample ID")
ax_unk.set_ylabel("Factors")
ax_unk.tick_params(axis="x", rotation=90)

cax.set_ylabel("Factor value")
plt.tight_layout()
plt.show()

## 5. Classify each unknown sample: Luminal A or Luminal B

We already know (from earlier statistics) that all 10 unknown samples are
either Luminal A or Luminal B. A simple logistic regression on the MOFA
factors — trained on the labelled cohort — gives us a first, quantitative
pass at this split, which we can combine with the visual read from Section 4.

In [ ]:
"""
------------------
YOUR CODE HERE
------------------
1. Extract the subtype labels from y_train, handling the case where 
   it is a DataFrame by pulling out the relevant column

2. Align the training samples so that only samples present in both 
   the MOFA factors and the subtype labels are kept

3. Train a logistic regression classifier on the aligned training data

4. Predict subtypes for the test samples and store the results as a 
   Series
"""

In [ ]:
"""
------------------
YOUR CODE HERE
------------------
1. Use the trained classifier to predict the probability of each 
   subtype for every test sample

2. Store the results as a DataFrame, with samples as rows and 
   subtypes as columns
"""


fig, ax = plt.subplots(figsize=(8, max(6, 0.25 * predicted_proba.shape[0])))

sns.heatmap(
    predicted_proba,
    cmap="viridis",
    vmin=0, vmax=1,
    linewidths=0.2,
    linecolor="white",
    cbar_kws={"label": "Predicted probability"},
    ax=ax,
)
ax.set_title("Predicted subtype probabilities (challenge samples)")
ax.set_xlabel("Subtype")
ax.set_ylabel("Sample ID")
ax.tick_params(axis="x", rotation=45)
ax.tick_params(axis="y", rotation=0)

plt.tight_layout()
plt.show()

predicted_proba

## 6. Explain each unknown sample: driving genes -> diseases

Knowing Luminal A vs B narrows things down but doesn't tell us *which*
patient a sample belongs to. For that we look at which genes most strongly
drive each unknown sample's position in factor space, then look up which
diseases those genes are associated with in a knowledge graph — so we can
match against the clinical notes at the top of the notebook.

We run this for **all 10** unknown samples (not just one), since we need to
resolve every `Pat_i`.

In [ ]:
def top_features_weighted_by_expression(
    model,
    X: pd.DataFrame,
    sample_id: str,
    factor: str,
    view: str,
    top_n: int = 20,
    standardize: bool = True,
):
    """Rank features for one sample/factor by |weight| * |expression|.

    A feature's contribution to a sample's factor value is roughly proportional
    to its MOFA weight times how far that sample's (standardized) expression is
    from the cohort mean. Sorting by this product surfaces the genes that most
    plausibly explain *why* this particular sample sits where it does on this factor.
    """
    # W: features x factors
    W = model.get_weights(views=view, df=True)
    w = W[factor]

    x = X.loc[sample_id]
    if standardize:
        # z-score each feature across samples so weight and expression are comparable.
        x = (x - X.mean(axis=0)) / X.std(axis=0).replace(0, pd.NA)

    impact = (w.abs() * x.abs()).dropna().sort_values(ascending=False)

    out = pd.DataFrame({
        "weight_abs": w.abs().reindex(impact.index),
        "x_abs": x.abs().reindex(impact.index),
        "impact": impact,
    })
    return out.head(top_n)

In [ ]:
# Which factors and view to explain each sample with. Factor1/2/4/6 were the
# ones that visually separated subtypes best in Section 4 — feel free to widen
# this set (or try 'proteomics'/'methylation') if your matches aren't conclusive.
factors_to_explain = ["Factor1", "Factor2", "Factor4", "Factor6"]
view = "transcriptomics"

# top_genes_by_sample: {sample_id -> list of top driving gene names}
top_genes_by_sample = {}

"""
------------------
YOUR CODE HERE
------------------
1. Loop over each unknown test sample

2. For each sample, loop over each MOFA factor of interest and 
   retrieve the top 25 most important genes for that factor

3. Collect all the genes across all factors for that sample, 
   removing duplicates and sorting them

4. Store the final gene list for each sample in top_genes_by_sample
"""

for sample_id, genes in top_genes_by_sample.items():
    print(f"{sample_id}: {len(genes)} candidate driving genes")

In [ ]:
# Load the gene-disease knowledge graph once.
G = load_kg()
print_graph_info(G)

In [ ]:
# For each unknown sample, map its driving genes onto the KG and pull the
# diseases most associated with them — these are the clues to match against
# the clinical notes at the top of the notebook.
disease_hits_by_sample = {}

"""
------------------
YOUR CODE HERE
------------------
1. Loop over each sample and its associated top genes

2. Map the genes to the knowledge graph and count how many are found

3. Retrieve the top 30 diseases associated with those genes from the 
   knowledge graph

4. Store the disease results and print a summary showing how many 
   genes were matched, then display the top 10 diseases
"""

## 7. Your answer: match each sample to a patient

Using:

- the predicted **Luminal A / Luminal B** subtype for each sample (Section 5), and
- the **top associated diseases** for each sample (Section 6),

match each of `Pat_0` ... `Pat_9` to one of the 10 `TCGA-xxxx` IDs from the
clinical notes at the top of this notebook. Every ID should be used exactly once,
and your final answer should contain 5 Luminal A and 5 Luminal B matches.

In [ ]:
# TODO: fill this in with your final answer, e.g. {"Pat_0": "TCGA-EW-A6S9", ...}
sample_to_patient = {pat_id: None for pat_id in pat_ids}

sample_to_patient

In [ ]:
# Sanity checks before you submit.
assert all(v is not None for v in sample_to_patient.values()), "Every Pat_i needs a match."
assert set(sample_to_patient.values()) == set(missing_ids), "Every TCGA ID should be used exactly once."

n_luma = (predicted_subtype.reindex(sample_to_patient.keys()) == "LumA").sum()
n_lumb = (predicted_subtype.reindex(sample_to_patient.keys()) == "LumB").sum()
print(f"Predicted split among your matches -> LumA: {n_luma}, LumB: {n_lumb} (expect 5 / 5)")

# Uncomment once you're happy with your answer:
# evaluate_predictions(sample_to_patient)